# parity

The acceptance test, and the last step of `run_pipeline`: the two engines built the same
models from the same files, so their gold tables must agree. This notebook reads both
engines' gold tables, the `<engine>_mart` schema, straight from OneLake as Delta, with
DuckDB, and fails the run if they differ. dbt is not involved, and neither engine grades its
own output.

- **`fct_summary`** is compared DAY BY DAY: rows, units (DUIDs) and the sum of `time` must be
  identical, and a date on one engine only is a failure. Totals over the whole table would
  let errors cancel out (a row credited to another unit, a time shifted by one interval), and
  a failure here names its date.
- **Money** sums (`mw`, `price`) get a relative tolerance of 1e-7 per day, because two
  engines can compute the same thing and still disagree in the last place: `DOUBLE ->
  DECIMAL` rounds HALF_UP on Spark and otherwise in T-SQL.
- **`dim_duid` and `dim_calendar`** are small, and compared row by row.

The tolerance absorbs rounding. It is not there to paper over a logic difference: a leg
that drifts past it is a finding, not a threshold to raise.

DuckDB is a library here, preinstalled with the notebook's own OneLake credentials, so
nothing is installed and no token is handled.

In [ ]:
# 1. SETTINGS, and where each engine's gold layer is: its item, and the schema
# macros/generate_schema_name.sql gives it.
import json
import urllib.request

import duckdb
import notebookutils

ENGINES = ("dwh", "spark")
ITEMS = {"dwh": ("warehouses", "dbt_dwh"), "spark": ("lakehouses", "dbt")}
REL_TOL = 1e-7

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
if vl.engines != "all":
    notebookutils.notebook.exit(f"parity skipped: only {vl.engines} was built")

workspace = notebookutils.runtime.context["currentWorkspaceId"]


def item(kind, name):
    """An item's id, by name, from the Fabric REST API, as whoever runs the notebook."""
    request = urllib.request.Request(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace}/{kind}",
        headers={"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"})
    with urllib.request.urlopen(request) as r:
        found = [i["id"] for i in json.loads(r.read())["value"] if i["displayName"] == name]
    if not found:
        raise RuntimeError(f"{name} is not in this workspace; install the project first")
    return found[0]


def schema(engine):
    return f"{engine}_mart" if vl.dbt_schema == "mart" else f"{vl.dbt_schema}_{engine}_mart"


ROOT = {e: f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{item(*ITEMS[e])}/Tables/{schema(e)}"
        for e in ENGINES}
for e in ENGINES:
    print(f"{e}: {ROOT[e]}")

In [ ]:
# 2. READ both gold layers as Delta: one view per engine and table, e.g. dwh_fct_summary.
TABLES = ("fct_summary", "dim_duid", "dim_calendar")

con = duckdb.connect()
# Dates and timestamps read the same from both, whatever the session's zone.
con.sql("SET TimeZone = 'UTC'")
for e in ENGINES:
    for t in TABLES:
        con.sql(f"CREATE VIEW {e}_{t} AS SELECT * FROM delta_scan('{ROOT[e]}/{t}')")
failures = []

In [ ]:
# 3. fct_summary, DAY BY DAY.
EXACT = ("rows", "duids", "time_sum")
APPROX = ("mw_sum", "price_sum")


def per_day(engine):
    query = ('SELECT "date", COUNT(*), COUNT(DISTINCT DUID), SUM(CAST("time" AS BIGINT)), '
             'CAST(SUM(mw) AS DOUBLE), CAST(SUM(price) AS DOUBLE) '
             f'FROM {engine}_fct_summary GROUP BY "date"')
    return {row[0]: row[1:] for row in con.sql(query).fetchall()}


days = {e: per_day(e) for e in ENGINES}
a_name, b_name = ENGINES
a, b = days[a_name], days[b_name]
for day in sorted(a.keys() | b.keys()):
    if day not in a or day not in b:
        failures.append(f"fct_summary {day}: only on {a_name if day in a else b_name}")
        continue
    for i, field in enumerate(EXACT):
        if a[day][i] != b[day][i]:
            failures.append(f"fct_summary {day} {field}: {a_name}={a[day][i]} {b_name}={b[day][i]}")
    for i, field in enumerate(APPROX, len(EXACT)):
        x, y = a[day][i] or 0.0, b[day][i] or 0.0
        diff = abs(x - y) / max(abs(x), abs(y), 1.0)
        if diff > REL_TOL:
            failures.append(f"fct_summary {day} {field}: {a_name}={x!r} {b_name}={y!r} "
                            f"(relative difference {diff:.3e} > {REL_TOL:.0e})")

for e in ENGINES:
    print(f"fct_summary on {e}: {sum(r[0] for r in days[e].values()):,} rows, {len(days[e])} days")

In [ ]:
# 4. THE DIMENSIONS, ROW BY ROW. By column NAME, not position: the engines need not order
# the columns alike.
for t in ("dim_duid", "dim_calendar"):
    columns = {e: [c.lower() for c in con.sql(f"SELECT * FROM {e}_{t} LIMIT 0").columns]
               for e in ENGINES}
    if set(columns[a_name]) != set(columns[b_name]):
        failures.append(f"{t}: columns differ: {sorted(set(columns[a_name]) ^ set(columns[b_name]))}")
        continue
    select = ", ".join(f'"{c}"' for c in columns[a_name])
    for this, that in ((a_name, b_name), (b_name, a_name)):
        extra = con.sql(f"SELECT {select} FROM {this}_{t} EXCEPT ALL "
                        f"SELECT {select} FROM {that}_{t}").fetchall()
        failures += [f"{t}: only on {this}: {row}" for row in extra[:5]]
        if len(extra) > 5:
            failures.append(f"{t}: and {len(extra) - 5} more rows only on {this}")
    counts = {e: con.sql(f"SELECT COUNT(*) FROM {e}_{t}").fetchone()[0] for e in ENGINES}
    print(f"{t}: " + ", ".join(f"{n:,} rows on {e}" for e, n in counts.items()))

In [ ]:
# 5. VERDICT
if failures:
    # Before changing a tolerance, check the two usual causes:
    #   * a string join key with a trailing space. T-SQL pads on comparison, Spark does
    #     not, and that changes row counts, not just sums.
    #   * one engine's model drifting from the shared logic. Diff models/aemo/dwh/ against
    #     models/aemo/spark/.
    shown = failures[:50] + ([f"and {len(failures) - 50} more"] if len(failures) > 50 else [])
    raise RuntimeError("PARITY FAILED, the engines did not produce the same gold layer:\n  "
                       + "\n  ".join(shown))

print(f"\nPARITY OK: {a_name} and {b_name} agree on {', '.join(TABLES)}")